# TP1 — Retrieval-Augmented Generation (RAG) sobre CVs

**Autor:** Luciano  
**Curso:** CEIA — FIUBA  
**Stack:** Pinecone (vector store) · Groq (LLM) · HuggingFace (embeddings) · LangChain (orquestación)

## Objetivo

Implementar un chatbot basado en RAG que responda preguntas sobre un corpus de CVs en PDF. El sistema debe:
1. Indexar los CVs en una base vectorial.
2. Recuperar fragmentos relevantes para cada consulta.
3. Generar respuestas condicionadas al contexto, citando las fuentes.

## ¿Por qué RAG?

Un LLM por sí solo no conoce los CVs del corpus y alucinaría respuestas. RAG resuelve esto inyectando en el prompt solo los fragmentos relevantes recuperados de una base vectorial, anclando la generación en evidencia. Es más barato que fine-tuning y se actualiza agregando documentos al índice.

## 0. Setup

Instalá las dependencias (`pip install -r requirements.txt`) y seteá las variables de entorno en `.env`:
- `GROQ_API_KEY`
- `PINECONE_API_KEY`

In [1]:
import sys
from pathlib import Path

# Permitir importar el paquete src/ desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from src import config
from src.ingestion import ingest
from src.rag_chain import build_chain, invoke, invoke_without_rag
from src.retriever import build_retriever, build_vectorstore
from src.evaluation import (
    EvalQuery, evaluate_all, summary_metrics,
    precision_at_k, recall_at_k, reciprocal_rank,
)

print(f"Corpus:     {config.CVS_DIR}")
print(f"Index:      {config.PINECONE_INDEX_NAME}")
print(f"Namespace:  {config.PINECONE_NAMESPACE}")
print(f"Embeddings: {config.EMBED_MODEL}")
print(f"LLM:        {config.GROQ_MODEL}")

Corpus:     /home/lucianoceb/Documents/CEIA/CEIA-LLMIAG-main/ClaseVI/tp/rag-cvs/data/cvs
Index:      cv-rag
Namespace:  recursive
Embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
LLM:        llama-3.1-8b-instant


## 1. Arquitectura

```
┌──────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────┐
│  data/cvs/   │ → │ PyPDFLoader  │ → │  Splitter    │ → │ MiniLM   │
│  *.pdf       │   │ + limpieza   │   │ (500/50 ó    │   │ multilg  │
│              │   │              │   │  semántico)  │   │ 384 dim  │
└──────────────┘   └──────────────┘   └──────────────┘   └────┬─────┘
                                                                │
                                                        ┌───────▼──────┐
                                                        │   Pinecone   │
                                                        │ (serverless) │
                                                        └───────┬──────┘
                                                                │
┌──────────┐   ┌──────────────┐   ┌──────────────────┐   ┌──────▼──────┐
│  query   │ → │ (reformular  │ → │   retrieve       │ ← │  index      │
│          │   │ si hay hist) │   │    top-k         │   │             │
└──────────┘   └──────────────┘   └────────┬─────────┘   └─────────────┘
                                           │
                                  ┌────────▼──────────┐
                                  │  Groq (Llama 3.1) │
                                  │  + system prompt  │
                                  └────────┬──────────┘
                                           │
                                  ┌────────▼──────────┐
                                  │ respuesta citada  │
                                  └───────────────────┘
```

**Decisiones:**
- **Embeddings multilingües locales** (`paraphrase-multilingual-MiniLM-L12-v2`, 384 dim): CVs en español, corre gratis.
- **Pinecone serverless**: vector store managed, escala a miles de CVs sin infra.
- **Groq + Llama 3.1 8B**: latencia baja y gratis en tier free. Alternativa: `llama-3.3-70b-versatile` para mayor calidad.
- **Chunk size 500 / overlap 50**: secciones de CV son cortas; chunks más grandes diluyen la señal.
- **History-aware retriever**: reformula queries de follow-up usando historial antes de buscar.

## 2. Ingesta

Cargamos los PDFs, los limpiamos (fix del artefacto `s t r u c t u r e` -> `structure`), los partimos en chunks y los subimos a Pinecone.

Esta celda llama a la API de Pinecone y puede tardar ~30s-1min según la cantidad de CVs.

In [ ]:
"""n_chunks = ingest(
    cvs_dir=str(config.CVS_DIR),
    force=True,           # limpia vectores previos para evitar duplicados
    use_semantic=False,   # True = SemanticChunker (mismo MiniLM)
)
print(f"\n {n_chunks} chunks indexados.")"""

## 3. Inspección del retriever

Antes de llamar al LLM, verificamos que el retrieval funciona. Hacemos una query y miramos qué chunks devuelve y con qué score.

Utilizamos MiniLM, que produce vectores de 384 dimensiones entrenados con objetivos de similitud semántica multilingüe.

In [2]:
vectorstore = build_vectorstore()

query = "¿Qué candidatos tienen experiencia con Python y machine learning?"
results = vectorstore.similarity_search_with_score(query, k=4)

for i, (doc, score) in enumerate(results, 1):
    source = doc.metadata.get('source', '?')
    page = doc.metadata.get('page', '?')
    print(f"\n[{i}] score={score:.3f} | {source} (pág. {page})")
    print('-' * 70)
    print(doc.page_content[:300], '...')

23:51:17 [INFO] Use pytorch device_name: cuda
23:51:17 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
23:51:20 [INFO] Discovering subpackages in _NamespacePath(['/home/lucianoceb/Documents/CEIA/CEIA-LLMIAG-main/ClaseVI/tp/rag-cvs/.venv/lib/python3.12/site-packages/pinecone_plugins'])
23:51:20 [INFO] Looking for plugins in pinecone_plugins.inference
23:51:20 [INFO] Installing plugin inference into Pinecone



[1] score=0.480 | cv1.pdf (pág. 1.0)
----------------------------------------------------------------------
de Tiempo & Forecasting, Detección del Fraude, Procesamiento del Lenguaje Natural
(NLP), Computer vision.
Herramientas y Tecnologías:Python, SQL, PowerBi, AWS (para data engineering),
Docker, Airflow, MLFlow, Databricks.
Bibliotecas (Python):numpy, pandas, matplotlib, seaborn, scipy, scikit-learn, n ...

[2] score=0.439 | cv1.pdf (pág. 1.0)
----------------------------------------------------------------------
Desarrollo de algoritmos para la gestión eficiente de reclamos (Random Forest, Series de
Tiempo, Clustering).
Elaboración de tableros e informes para diferentes partes interesadas.
Formación Académica
Especialista en Inteligencia Artificial | Universidad Nacional de Buenos Aires
(Facultad de Ingenie ...

[3] score=0.430 | cv1.pdf (pág. 0.0)
----------------------------------------------------------------------
AI Engineer. Actualmente culminando la especialización en Intel

**Nota sobre el score:** Pinecone con métrica `cosine` devuelve similitud coseno directa en rango [-1, 1]; valores cercanos a 1 son chunks muy relevantes y típicamente los primeros resultados están entre 0.5 y 0.8.

## 4. Pipeline end-to-end

Construimos la cadena completa (retrieve → prompt → Groq → parse) y la probamos.

In [3]:
chain, retriever, llm = build_chain()

result = invoke(query, chain, retriever)
print(result.answer)
print("\n📎 Fuentes recuperadas:")
for d in result.source_documents:
    print(f"  - {d.metadata.get('source')} (pág. {d.metadata.get('page')})")

23:51:22 [INFO] Use pytorch device_name: cuda
23:51:22 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
23:51:25 [INFO] Discovering subpackages in _NamespacePath(['/home/lucianoceb/Documents/CEIA/CEIA-LLMIAG-main/ClaseVI/tp/rag-cvs/.venv/lib/python3.12/site-packages/pinecone_plugins'])
23:51:25 [INFO] Looking for plugins in pinecone_plugins.inference
23:51:25 [INFO] Installing plugin inference into Pinecone
23:51:28 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Según el contexto, los siguientes candidatos tienen experiencia con Python y machine learning:

- El candidato menciona en [Fragmento 1 | fuente: cv1.pdf | página: 1.0] que utiliza Python como herramienta y también menciona bibliotecas como numpy, pandas, matplotlib, seaborn, scipy, scikit-learn, nltk, opencv y pytorch.
- El candidato menciona en [Fragmento 2 | fuente: cv1.pdf | página: 1.0] que ha desarrollado algoritmos para la gestión eficiente de reclamos utilizando Random Forest, Series de Tiempo y Clustering, lo que implica experiencia con machine learning.
- El candidato menciona en [Fragmento 3 | fuente: cv1.pdf | página: 0.0] que ha aplicado ciencia de datos e implementado algoritmos de machine learning para generar valor estratégico, y también menciona proyectos de machine learning destacados, como el desarrollo de pronósticos para predecir kilos de productos vendidos por punto de venta.
- El candidato menciona en [Fragmento 4 | fuente: cv1.pdf | página: 0.0] que ha desarroll

In [4]:
# Otra pregunta más específica
r = invoke("¿Alguno trabajó en una startup de fintech?", chain, retriever)
print(r.answer)

23:51:28 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


No encontré esa información en los CVs disponibles.


In [5]:
# Pregunta cuya respuesta NO está en el corpus: el sistema debe admitirlo
r = invoke("¿Cuál es la capital de Mongolia?", chain, retriever)
print(r.answer)

23:51:29 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


No encontré esa información en los CVs disponibles.


## 5. Conversación con follow-ups

Verificamos que el history-aware retriever funciona: si preguntamos algo ambiguo después de un contexto, debería reformular internamente.

In [6]:
from langchain_core.messages import HumanMessage, AIMessage

history = []

q1 = "¿Qué estudió Ana García?"
r1 = invoke(q1, chain, retriever, chat_history=history)
print(f"Q1: {q1}\nA1: {r1.answer}\n")
history.extend([HumanMessage(content=q1), AIMessage(content=r1.answer)])

q2 = "¿Y dónde trabaja ahora?"  # pregunta ambigua, necesita el contexto
r2 = invoke(q2, chain, retriever, chat_history=history)
print(f"Q2: {q2}\nA2: {r2.answer}")

23:51:30 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Q1: ¿Qué estudió Ana García?
A1: No encontré esa información en los CVs disponibles.



23:51:31 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
23:51:31 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Q2: ¿Y dónde trabaja ahora?
A2: No encontré esa información en los CVs disponibles.


## 6. Evaluación

Evaluamos dos aspectos:

### 6.1 Métricas de retrieval

Definimos manualmente un set de queries con *ground truth*: para cada pregunta, qué archivos contienen la respuesta correcta. Métricas clásicas de IR:

- **Precision@k**: de los k chunks recuperados, fracción que son relevantes.
- **Recall@k**: de todos los documentos relevantes, fracción recuperada en los top-k.
- **MRR (Mean Reciprocal Rank)**: posición del primer relevante. Premia que lo bueno esté arriba.

Cada `relevant_sources` debe listar los filenames (no paths) de los CVs que contienen la respuesta correcta.

In [7]:
eval_set = [
    EvalQuery(
        question="¿Qué candidatos tienen experiencia técnica con Docker?",
        relevant_sources=["cv1.pdf", "cv3.pdf", "cv4.pdf"],
    ),
    EvalQuery(
        question="¿Quiénes poseen una formación de posgrado finalizada o en curso en el área de Inteligencia Artificial?",
        relevant_sources=["cv1.pdf", "cv4.pdf"],
    ),
    EvalQuery(
        question="¿Qué candidato tiene experiencia específica en el despliegue de redes 5G?",
        relevant_sources=["cv2.pdf"],
    ),
    EvalQuery(
        question="¿Qué candidatos cuentan con un nivel de inglés avanzado (C1 o superior)?",
        relevant_sources=["cv2.pdf", "cv3.pdf"],
    ),
    EvalQuery(
        question="¿Quién tiene experiencia en el sector de consumo masivo o franquicias (ej. Grido)?",
        relevant_sources=["cv1.pdf"],
    ),
    EvalQuery(
        question="¿Qué perfiles mencionan explícitamente el uso de Kubernetes para la orquestación?",
        relevant_sources=["cv2.pdf", "cv3.pdf"],
    ),
    EvalQuery(
        question="¿Quiénes tienen experiencia docente o académica (ayudantía o instructor)?",
        relevant_sources=["cv1.pdf", "cv2.pdf"],
    ),
    EvalQuery(
        question="¿Qué candidatos dominan el lenguaje de programación Go?",
        relevant_sources=["cv3.pdf"],
    ),
    EvalQuery(
        question="¿Quiénes han trabajado con herramientas de MLOps como MLflow o Kubeflow?",
        relevant_sources=["cv1.pdf", "cv4.pdf"],
    ),
    EvalQuery(
        question="¿Qué candidatos son Ingenieros Industriales de profesión?",
        relevant_sources=["cv1.pdf"],
    ),
    EvalQuery(
        question="¿Qué perfiles demuestran experiencia en el desarrollo de APIs o microservicios?",
        relevant_sources=["cv3.pdf", "cv4.pdf"],
    ),
    EvalQuery(
        question="¿Quién tiene experiencia en análisis de sentimientos o NLP?",
        relevant_sources=["cv1.pdf", "cv4.pdf"],
    )
]

print(f"Set de evaluación con {len(eval_set)} queries generado exitosamente.")

Set de evaluación con 12 queries generado exitosamente.


In [8]:
K = 4
df_eval = evaluate_all(
    eval_set=eval_set,
    chain=chain,
    retriever=retriever,
    llm=llm,
    k=K,
    run_no_rag_baseline=True,
)
df_eval[['question', f'precision@{K}', f'recall@{K}', 'mrr']]

[1/12] ¿Qué candidatos tienen experiencia técnica con Docker?...


23:51:43 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
23:51:45 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[2/12] ¿Quiénes poseen una formación de posgrado finalizada o en curso en el ...


23:51:45 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:51:45 [INFO] Retrying request to /openai/v1/chat/completions in 1.000000 seconds
23:51:46 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:51:46 [INFO] Retrying request to /openai/v1/chat/completions in 4.000000 seconds
23:51:51 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
23:51:52 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[3/12] ¿Qué candidato tiene experiencia específica en el despliegue de redes ...


...

[11/12] ¿Qué perfiles demuestran experiencia en el desarrollo de APIs o micros...


23:53:58 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:53:58 [INFO] Retrying request to /openai/v1/chat/completions in 10.000000 seconds
23:54:09 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:54:09 [INFO] Retrying request to /openai/v1/chat/completions in 2.000000 seconds
23:54:11 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
23:54:11 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:54:11 [INFO] Retrying request to /openai/v1/chat/completions in 1.000000 seconds
23:54:14 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


[12/12] ¿Quién tiene experiencia en análisis de sentimientos o NLP?...


23:54:15 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:54:15 [INFO] Retrying request to /openai/v1/chat/completions in 12.000000 seconds
23:54:27 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 429 Too Many Requests"
23:54:27 [INFO] Retrying request to /openai/v1/chat/completions in 3.000000 seconds
23:54:30 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
23:54:31 [INFO] HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


,question,precision@4,recall@4,mrr
0,¿Qué candidatos tienen experiencia técnica con...,1.000000,0.666667,1.0
1,¿Quiénes poseen una formación de posgrado fina...,1.000000,1.000000,1.0
2,¿Qué candidato tiene experiencia específica en...,0.500000,1.000000,1.0
3,¿Qué candidatos cuentan con un nivel de inglés...,0.000000,0.000000,0.0
4,¿Quién tiene experiencia en el sector de consu...,0.500000,1.000000,1.0
5,¿Qué perfiles mencionan explícitamente el uso ...,0.000000,0.000000,0.0
6,¿Quiénes tienen experiencia docente o académic...,0.500000,0.500000,1.0
7,¿Qué candidatos dominan el lenguaje de program...,0.333333,1.000000,0.5
8,¿Quiénes han trabajado con herramientas de MLO...,1.000000,1.000000,1.0
9,¿Qué candidatos son Ingenieros Industriales de...,0.333333,1.000000,1.0


In [9]:
summary = summary_metrics(df_eval, k=K)
for metric, value in summary.items():
    if isinstance(value, float):
        print(f"{metric}: {value:.3f}")
    else:
        print(f"{metric}: {value}")

precision@4_mean: 0.569
recall@4_mean: 0.764
mrr_mean: 0.792
n_queries: 12


### 6.2 Comparación cualitativa: RAG vs no-RAG

Comparamos las respuestas del sistema RAG con las del mismo LLM sin contexto. El LLM sin contexto no puede saber nada sobre estos CVs: debería admitir desconocimiento o alucinar.

In [10]:
for i, row in df_eval.head(3).iterrows():
    print(f"\n{'='*80}")
    print(f"❓ {row['question']}")
    print('='*80)
    print(f"\n✅ Con RAG:\n{row['rag_answer']}")
    print(f"\n❌ Sin RAG (baseline):\n{row['no_rag_answer']}")


❓ ¿Qué candidatos tienen experiencia técnica con Docker?

✅ Con RAG:
No encontré esa información en los CVs disponibles.

❌ Sin RAG (baseline):
Excelente pregunta! La experiencia técnica con Docker es una habilidad muy valiosa en la industria actual. Aquí te presento algunos candidatos que podrían tener experiencia técnica con Docker:

1. **Desarrolladores de software**: Los desarrolladores de software con experiencia en contenedores y orquestación de aplicaciones pueden ser candidatos ideales para trabajar con Docker.
2. **Ingenieros de sistemas**: Los ingenieros de sistemas con experiencia en administración de sistemas, seguridad y optimización de rendimiento pueden ser candidatos adecuados para trabajar con Docker.
3. **Arquitectos de sistemas**: Los arquitectos de sistemas con experiencia en diseño y implementación de soluciones de contenedores y orquestación de aplicaciones pueden ser candidatos valiosos para trabajar con Docker.
4. **Administradores de infraestructura**: Los adm

## 7. Conclusiones

- El sistema RAG recupera información relevante de los CVs y genera respuestas ancladas en evidencia, citando las fuentes entre corchetes.
- El LLM sin contexto (no-RAG) no puede responder sobre candidatos específicos; el delta de utilidad es enorme y cualitativamente obvio.
- Las métricas de retrieval (precision@k, recall@k, MRR) permiten diagnosticar si las fallas están en la recuperación o en la generación — un precision@k bajo indica problemas de retrieval; una respuesta mala con precision alto indica problema de generación.